In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display

import gc
import itertools

```python
"""
Paths:
Chunk0950                    /Chunk0950/001_007/AO2Dtree.root
Chunk1000                    [ /Chunk1000/001_005/AO2Dtree.root, /Chunk1000/006_009/AO2Dtree.root ]
Chunk1010                    [ /Chunk1010/001_005/AO2Dtree.root, /Chunk1010/006_010/AO2Dtree.root ]
Chunk1020                    /Chunk1020/001_007/AO2Dtree.root
Chunk1030                    [ /Chunk1030/001_005/AO2Dtree.root, /Chunk1030/006_011/AO2Dtree.root ]
Chunk1040                    /Chunk1040/001_007/AO2Dtree.root
Chunk1050                    [ /Chunk1050/001_005/AO2Dtree.root, /Chunk1050/006_010/AO2Dtree.root ]
Chunk1100                    [ /Chunk1100/001_005/AO2Dtree.root, /Chunk1100/006_009/AO2Dtree.root ]
Chunk1110                    /Chunk1110/001_005/AO2Dtree.root
Chunk1120                    [ /Chunk1120/001_005/AO2Dtree.root, /Chunk1120/006_011/AO2Dtree.root ]
Chunk1130                    /Chunk1130/001_008/AO2Dtree.root
Chunk1140                    [ /Chunk1140/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1150                    /Chunk1150/001_007/AO2Dtree.root
Chunk1200                    [ /Chunk1200/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1210                    /Chunk1210/001_008/AO2Dtree.root
Chunk1220                    /Chunk1220/001_008/AO2Dtree.root
Chunk1300                    [ /Chunk1300/001_006/AO2Dtree.root, /Chunk1300/003/AO2Dtree.root, /Chunk1300/007_010/AO2Dtree.root ]
"""
```

In [2]:
#base_path = "../../mnt/SingleTrackTrees/Data/alice_data_2023_LHC23f_535087_apass4_1300_Thinner/Output/Run535087"
base_path = "~/Downloads"
which_chunk = "Chunk0950"
which_number = "001_007"
which_file = "AO2Dtree.root"
path = base_path + "/".join(["/", which_chunk, which_number, which_file])
#path = "~/Downloads/AO2Dtree.root"
file = uproot.open(path)

In [3]:
# !!! NEW CUTS !!!
'''
This is the equivalent for the cut for Kaons

(real_data["fNsigmaTPCka"].abs() < 2) & \
          ( 
              ( (real_data["fPt"] <= 1.4) & (real_data["fNsigmaTOFka"].abs() <= 4)  ) |
              ( (real_data["fPt"].between(1.4,2.0)) & (real_data["fNsigmaTOFka"].abs() <= 2) )
          )
'''

# Kaon hypothesis
Ka_hyp = " ( fNsigmaTPCka.abs() < 2 ) & "\
         " ( \
             ( (fPt <= 1.4) & (fNsigmaTOFka.abs() <=4) ) | \
             ( (fPt.between(1.4,2.0)) & (fNsigmaTOFka.abs() <= 2) ) \
            )"

'''
The following is equivalent to this cut for Pions
(real_data["fNsigmaTPCpi"].abs() < 2) & \
          ( 
              ( (real_data["fPt"] <= 1.4) & (real_data["fNsigmaTOFpi"].abs() <= 4)  ) |
              ( (real_data["fPt"].between(1.4,2.0)) & (real_data["fNsigmaTOFpi"].abs() <= 2) )
          )
'''

# Pion Hypothesis
Pi_hyp = " ( fNsigmaTPCpi.abs() < 2 ) & "\
         " ( \
             ( (fPt <= 1.4) & (fNsigmaTOFpi.abs() <=4) ) | \
             ( (fPt.between(1.4,2.0)) & (fNsigmaTOFpi.abs() <= 2) )\
            )"

cut_expression = f"({Ka_hyp} | {Pi_hyp})"

out_name = f"Cuts_for_{which_chunk}_{which_number}.txt"
with open(out_name, "w") as f:
  f.write(cut_expression)

In [4]:
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    # angle = -row["fAlpha"]
    # rot = np.array([[np.cos(angle),np.sin(angle)], [-np.sin(angle), np.cos(angle)]]) # inverse matrix of the one from to_track_coord (-alfa)
    # Rewriting the above matrix using cosine/sin even/odd function properties: less multiplications
    angle = row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    xy_out = rot.dot(xy_in)
    return( xy_out )

def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * (x_SV - x1) + row1["fZ"]
    z2_track = pz2/px2 * (x_SV - x2) + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

In [5]:
names_dirs = file.keys(filter_classname="TDirectory")
subsets = np.array_split(range(0,len(names_dirs)),4)
subsets
len(subsets)

4

In [6]:
# Load T-Trees from directories
names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")
names_track      = file.keys(filter_name=r"*O2filtertrack")
names_coll       = file.keys(filter_name=r"*O2collision_001")

# number of subsets to create: this is due to memory problems when doing the combinatorial
N_SPLITS = 4
subsets = np.array_split(range(0,len(names_coll )),N_SPLITS)

collision_offset = 0          # variable to correct the index of the collision (because it starts from 0 at each new TTree)

# Particle masses in GeV
m_K = 0.493677
m_pi = 0.139570
m_d0 = 1.86484

for J in range (len(subsets)):
    
    list_of_df = []               # add the dataframes in a list (we will concat them later)
    
    for i in subsets[J]:
    
        # Read collision tree
        df_coll = file[ names_coll[i] ].arrays(["fPosX", "fPosY", "fPosZ"], library="pd")   # I take the fPosZ column as a DataFrame
        # list_of_collision_df.append(df_coll)               # add the dataframe in a list (we will concat them later)
    
        # # Read track and trackextr using boolean mask for track and trackextr: NEW: removed TOF/TPC for proton
        df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY",
             "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTOFpi", "fNsigmaTOFka"], library="pd" )
        df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
        # merge all rows
        df_track = pd.merge(left=df_track, right=df_coll, how="inner", left_on = "fIndexCollisions", right_index=True)
        df_trackextr = pd.merge(left=df_trackextr, right=df_track, how='inner', left_index=True, right_index=True)
        
        # NEW : Assign particle nature hypothesis based on the chosen cuts
        mask_both = df_trackextr.eval(f"({Ka_hyp} & {Pi_hyp})")
        mask_Ka = df_trackextr.eval(Ka_hyp)
        mask_Pi = df_trackextr.eval(Pi_hyp)
        # From the documentation of numpy select:
        # numpy.select(condlist, choicelist, default=0) When multiple conditions are satisfied, the first one encountered in condlist is used.
        df_trackextr["Hyp"] = np.select([mask_both, mask_Ka, mask_Pi], ["Both", "Kaon", "Pion"], default="Bkg")
        mask = mask_Ka | mask_Pi        # create boolean mask
        # Apply the SAME filter to df_trackextr and df_track to keep them aligned (and keep only useful columns):
        df_trackextr = df_trackextr.loc[mask, ["fIndexCollisions","fAlpha", "fX", "fY", "fZ","fPt", "fEta", "fCharge", "fDcaXY",
                                               "Hyp", "fPosX", "fPosY", "fPosZ"] ]
    
        # the rows where the fIndexCollision is negative have been excluded thanks to the merge on the index of collision, which can only be
        # non-negative. So, we keep only those with |fPosZ| < 10
        valid = df_trackextr["fPosZ"].abs() < 10
        df_trackextr = df_trackextr[valid].reset_index(drop=True)
 
        df_trackextr["fIndexCollisions"] += collision_offset   # Fix local fIndexCollisions → global index 
    
        # save results:
        # # ALTERNATIVE 2:
        list_of_df.append( df_trackextr )                  # add the dataframe in a list (we will concat them later)
        
        # Update offset for next loop
        collision_offset += len(df_coll)     
    
        # let's free the memory RAM of unused dataframes:
        del df_trackextr
        del df_track
        gc.collect()
    
    
    # # UNCOMMENT FOR ALTERNATIVE 2:
    df = pd.concat(list_of_df, ignore_index=True)

    # Keep only those whose charge is 1 or -1
    df = df[df["fCharge"].isin([-1,1])]
    
    # Merge everything in the total dataframe
    N = len(df)

    # moment columns:
    df["px"] = df["fPt"] * np.cos(df["fAlpha"])
    df["py"] = df["fPt"] * np.sin(df["fAlpha"])
    df["pz"] = df["fPt"] * np.sinh(df["fEta"])
    
    # energy column (differentiating pions and kaons):
    mass = np.where(df["fCharge"] > 0, m_pi, m_K)
    anti_mass = np.where(df["fCharge"] > 0, m_K, m_pi)
    df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)
    df["Anti-Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + anti_mass**2)
    
    # Debug
    print("The starting dataframe has", len(df), "rows and ", len(df.columns), "columns.")
    memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The starting dataframe occupies {memory:.2f} MB")
    # df.head()

    # ALTERNATIVE 3:
    # let's initialize some lists, then we will create a dataframe
    collision_indices = []
    track1_indices = []
    track2_indices = []
    dcaXY_products = []
    inv_masses = []
    anti_masses = []
    pt_totals = []
    pz_totals = []
    SV_X = []
    SV_Y = []
    SV_Z = []
    decay_lengths = []
    cos_pointings = []
    mother = []
    
    counting=0 # debug variable
    
    # let's divide the dataframe for positive and negative charged
    df_pos = df[ df['fCharge']>0 ]
    df_neg = df[ df['fCharge']<0 ]
    
    # Iterate over each collision group
    for collision_idx in (df_neg['fIndexCollisions'].unique()):
        group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
        group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]
    
        # Only collisions with at least a pair
        if len(group_pos) < 1:   continue
    
        # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'
        group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
        group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})
    
        # let's crate indexes for all possible pairs:
        combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )
    
        # Iterate over all unique pairs of tracks
        for combo in combinat:
            row_neg = group_neg.iloc[combo[0]]
            row_pos = group_pos.iloc[combo[1]]
    
            product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
            
            # INVARIANT MASS calculation
            pt1, pt2 = row_neg['fPt'], row_pos['fPt']
   
            # exact formula:
            E1 = row_neg['Ene']
            E2 = row_pos['Ene']
            px1 = row_neg["px"]
            py1 = row_neg["py"]
            pz1 = row_neg["pz"]
            px2 = row_pos["px"]
            py2 = row_pos["py"]
            pz2 = row_pos["pz"]
            inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )

            anti_E1 = row_neg['Anti-Ene']
            anti_E2 = row_pos['Anti-Ene']
            anti_inv_mass = np.sqrt( (anti_E1+anti_E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )

            mother_hyp = ""

            # NEW: Assign mass of the mother based on the hypothesis on the daughters
            # inv_mass has been calculated under the assumption that the positive particle is the Pion, so the mother is D0
            if (  ( (row_neg["Hyp"] in ["Kaon", "Both"]) & (row_pos["Hyp"] == "Pion") )  | \
                  ( (row_neg["Hyp"] == "Kaon" ) & (row_pos["Hyp"] in ["Pion", "Both"] ) )  ):
                mother_hyp = "D0"

            elif( ( (row_neg["Hyp"] == "Pion" ) & (row_pos["Hyp"] in ["Kaon", "Both"] ) )| \
                  ( (row_neg["Hyp"] in ["Pion", "Both"] ) & (row_pos["Hyp"]=="Kaon" ) ) ):
                mother_hyp = "Anti-D0"

            else:
                mother_hyp = "Undecided"
    
    
            # total transverse momentum of the D0 candidate (used later for sliced plots)
            pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)
    
            # secondary vertex
            SV_coords = np.array( secondary_vertex(row_neg, row_pos) )
    
            # decay length: distance between PV and SV
            PV_coords = np.array( [row_pos["fPosX"], row_pos["fPosY"], row_pos["fPosZ"]] )
            decay_lengths.append( np.linalg.norm(SV_coords - PV_coords ) )
    
            # cosine of pointing angle: the latter is the angle between the direction of the mother particle and the line connecting PV and SV
            mother_direction = [px1+px2, py1+py2, pz1+pz2]
            flight_line = SV_coords - PV_coords
            cos_pointings.append ( np.dot(mother_direction, flight_line)/(np.linalg.norm(mother_direction)*np.linalg.norm(flight_line)) )
            
            # let's add the found pairs to the lists
            collision_indices.append(int(row_neg['fIndexCollisions']))
            dcaXY_products.append(product_dcaXY)
            inv_masses.append(inv_mass)
            anti_masses.append(anti_inv_mass)
            pt_totals.append(pt_total)
            pz_totals.append(pz1+pz2)
            mother.append(mother_hyp)
    
        # # let's free the memory RAM of unused dataframes:
        # # PROBLEM: THIS IS VERY SLOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # del group
        # gc.collect()
    
        # debug:
        counting += 1
        if counting % 50000 ==0: print(counting, end=' ')
    
    
    # create a dataframe with the result:
    df_pairs = pd.DataFrame({
        'collision_index': np.array(collision_indices, dtype="uint32"),
        'dcaXY_product': np.array(dcaXY_products, dtype="float32"),
        'inv_mass': np.array(inv_masses, dtype="float32"),
        'anti_mass': np.array(anti_masses, dtype="float32"),
        'pt': np.array(pt_totals, dtype="float32"),
        'pz': np.array(pz_totals, dtype="float32"),
        'decay_length': np.array(decay_lengths, dtype="float32"),
        'cos_pointing': np.array(cos_pointings, dtype="float32"),
        'particle': mother
    })

    # Debug
    print("The final dataframe has", len(df_pairs), "rows")
    memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The dataframe occupy {memory:.2f} MB")
    display( pd.concat([df_pairs.head(2),df_pairs.tail(3)]) )

    save_name = "pairs_" + "_".join([which_chunk, which_number, str(J)])
    df_pairs.to_pickle(save_name + "_v2.pkl")

    # free memory from the dataframes used for this subset
    del df
    del df_pairs
    del collision_indices 
    del dcaXY_products
    del inv_masses
    del pt_totals
    del pz_totals
    del decay_lengths
    del cos_pointings
    gc.collect()

    print(f"Iteration {J+1} out of {N_SPLITS} done!")
    print(f"{save_name} created.")

The starting dataframe has 1868397 rows and  18 columns.
The starting dataframe occupies 229.86 MB
50000 100000 150000 200000 250000 300000 The final dataframe has 1335845 rows
The dataframe occupy 113.65 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle
0,5,2.598084e-06,2.084048,2.011341,1.862834,0.365490,0.003792,-0.786519,Anti-D0
1,5,2.165110e-06,2.071549,2.047000,0.409603,0.636738,0.012767,0.288180,Undecided
1335842,1397820,-2.500939e-05,1.568068,1.560709,0.237486,0.247536,0.015665,-0.034013,Undecided
1335843,1397820,-1.442237e-05,1.188278,1.228307,0.640393,-0.343898,0.004687,-0.900233,Undecided
1335844,1397821,8.487575e-07,1.540336,1.719579,1.571326,0.692316,0.003653,0.805818,Undecided


Iteration 1 out of 4 done!
pairs_Chunk0950_001_007_0 created.
The starting dataframe has 1845387 rows and  18 columns.
The starting dataframe occupies 227.03 MB
50000 100000 150000 200000 250000 300000 The final dataframe has 1317134 rows
The dataframe occupy 112.05 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle
0,1397837,-3.715788e-06,1.216439,1.148594,1.100295,-0.041451,0.006132,0.925323,Undecided
1,1397837,8.547444e-06,1.451480,1.384370,0.884685,-0.132030,0.007236,-0.666283,Undecided
1317131,2779784,8.456925e-07,1.573293,1.513714,0.328602,0.535721,0.022396,0.442893,Undecided
1317132,2779784,1.097942e-06,1.181108,1.217067,0.223476,0.069684,0.007270,0.225567,Undecided
1317133,2779789,1.801198e-04,0.949880,0.831616,1.602123,0.730797,1.236046,-0.974123,Undecided


Iteration 2 out of 4 done!
pairs_Chunk0950_001_007_1 created.
The starting dataframe has 1829911 rows and  18 columns.
The starting dataframe occupies 225.12 MB
50000 100000 150000 200000 250000 300000 The final dataframe has 1298640 rows
The dataframe occupy 110.48 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle
0,2779800,-0.000004,0.902638,0.850648,0.947562,-0.743246,0.010624,-0.587775,Undecided
1,2779807,-0.000010,1.477102,1.481607,0.860107,0.757773,0.007181,-0.141642,Undecided
1298637,4162880,-0.000006,1.192491,1.012544,1.428820,1.325019,0.007355,0.961970,Undecided
1298638,4162880,0.000001,0.953810,0.857079,2.081013,1.266710,0.014618,-0.992584,Undecided
1298639,4162880,0.000016,1.918924,1.831127,1.224787,0.630114,0.008588,0.702737,Undecided


Iteration 3 out of 4 done!
pairs_Chunk0950_001_007_2 created.
The starting dataframe has 1809817 rows and  18 columns.
The starting dataframe occupies 222.65 MB
50000 100000 150000 200000 250000 300000 The final dataframe has 1282619 rows
The dataframe occupy 109.12 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle
0,4162892,-0.000004,0.738452,0.711118,1.324147,0.190034,0.040030,0.999169,Undecided
1,4162895,-0.000005,2.106997,2.032633,1.653340,-0.254297,0.003361,-0.963838,Undecided
1282616,5531408,-0.000003,1.304075,1.251216,0.575702,-0.364575,0.007863,0.673443,Anti-D0
1282617,5531408,0.000001,1.374014,1.406647,0.409742,0.122337,0.003664,-0.404899,Undecided
1282618,5531408,-0.000004,1.493079,1.487441,0.588019,-0.304964,0.007060,0.769380,Anti-D0


Iteration 4 out of 4 done!
pairs_Chunk0950_001_007_3 created.
